# Predicting Upcoming-Iteration Peak Eqsat Memory

Compare models that predict the upcoming iteration's sampled peak from its
pre-search egraph, scheduler/search-pressure state, prior work, and live heap.

`generate.py` records per-iteration measurements for terms in
`data/seed_terms/*/terms.json`. Every byte count is the **absolute** process
live heap, the same coordinate system `--max-memory` uses, so readings are
directly comparable to a ceiling across runs and workers.

The evaluation reports peak growth, then scores the
model on the decision it actually drives: stopping a run before it breaks the
memory ceiling. Cross-validation is grouped by seed term.


In [ ]:
import altair as alt
import numpy as np
import polars as pl

import iteration_data as D
import memory_model as M
import memory_plots as MP
import plots as P

alt.theme.register("analysis", enable=True)(lambda: P.THEME)


## Load the iteration traces

Load the latest seed folder into one row per iteration. Rewrite counts are
stored in `rule_<name>` columns.

In [ ]:
SEED_DIR = D.resolve_seed_dir()

iterations = D.load_iterations(SEED_DIR)
iterations.select(
    "term_size", "iter_index", "egraph_nodes", "egraph_classes", "allocated", "n_rebuilds"
).head()

### Maximum egraph memory

Peak live heap for each term, with the median, 10th–90th percentile range,
and configured memory limit overlaid. Memory-limit stop iterations remain
visible because their transient peaks are the events the model must catch.

In [ ]:
MP.maximum_egraph_memory(iterations, SEED_DIR)


## Build the supervised frame

Each row is the online decision point immediately before upcoming iteration
`k`: current allocation/egraph/scheduler state come from `k`, previous work
comes from completed iteration `k-1`, and the target is the sampled peak in
`k`. Memory-limit stop iterations remain trainable because their peak is the
positive example of interest; pre-work partial stops are explicitly excluded.


In [ ]:
all_decision_rows = D.build_decision_rows(iterations)
transitions = all_decision_rows.filter(pl.col("target_trainable"))

SCALARS, RULE_FEATURES = D.feature_columns(transitions)
RAW_RULES = D.rules_from_frame(transitions)
FEATURES = SCALARS + RULE_FEATURES
print(f"{len(SCALARS)} scalar features, {len(RULE_FEATURES)} per-rule features")

transitions.select(
    "term_size",
    "upcoming_iter_index",
    "egraph_nodes",
    "allocated",
    "iteration_peak_allocated",
    "y_log_peak_growth",
).head()


## Target distribution

Distribution of the log upcoming peak-growth ratio.

In [ ]:
MP.growth_histogram(transitions)

## Cross-validated model comparison

Three predictors are evaluated with 5-fold `GroupKFold` by seed term:

- **naive (carry forward):** current memory
- **ridge:** log-scaled features
- **gradient boosting:** raw features

`median error ×` is the exponentiated median absolute log error.

In [ ]:
metrics, predictions = M.evaluate(transitions, FEATURES, RULE_FEATURES)
metrics

### Do the per-rule features help?

This ablation holds the gradient-boosting estimator and grouped CV folds constant,
adding aggregate scheduler state, per-rule active identity, then effective limits.

In [ ]:
ablation_regression, ablation_replay = M.scheduler_feature_ablation(
    all_decision_rows, RAW_RULES, (64 << 20, 128 << 20, 256 << 20, 500 << 20)
)
ablation_regression

In [ ]:
MP.metric_bars(metrics, metric="R2")

In [ ]:
MP.predicted_vs_actual(predictions, "log upcoming peak-growth ratio")

In [ ]:
MP.residual_distribution(predictions, "log upcoming peak-growth ratio")

Residuals by egraph size.

In [ ]:
MP.residual_vs_size(predictions, "log upcoming peak-growth ratio")

## Permutation importance

The boosted model is fitted on four group folds and evaluated on the fifth.

In [ ]:
importance = M.importances(transitions, FEATURES, RULE_FEATURES, target="y_log_peak_growth")
MP.importance_bars(importance)

## Catching ceiling breaks

The model exists to stop a run before it crosses `--max-memory`, so growth
accuracy is a means, not the goal. This section scores the predictions as the
Rust hook uses them: stop when
`allocated * exp(prediction + margin) >= ceiling`.

Only the *first* predicted stop in a run counts, because the hook halts the run
there; everything after it describes a future that never happens. Crossings are
therefore counted per run, not per row.

Two boundaries are compared. **raw** trusts the prediction as-is. **conservative**
adds the safety margin, the 99th percentile of held-out residuals, which shifts
predictions up to cover the runs the model underestimates.


In [ ]:
CEILINGS = (64 << 20, 128 << 20, 256 << 20, 500 << 20)

decisions, SAFETY_MARGIN = M.ceiling_sweep(all_decision_rows, FEATURES, RULE_FEATURES, CEILINGS)
print(f"safety margin {SAFETY_MARGIN:.4f} log-growth (x{np.exp(SAFETY_MARGIN):.3f})")
decisions


`recall` is the share of crossing runs stopped in time; `precision` is the share
of stopped runs that really would have crossed. `iters_warning_*` is how many
iterations of advance notice the catches gave.


In [ ]:
MP.ceiling_decisions_chart(decisions)


### How rare is a crossing?

Crossings are a small fraction of rows, and rarer the higher the ceiling. This
is why the aggregate regression scores above look strong while the decision that
matters can still be missed: a model can fit the bulk of ordinary growth well
and still miss the tail where memory runs away.


In [ ]:
base_rates = pl.DataFrame(
    [
        {
            "ceiling_mib": c / 2**20,
            "rows_below": int(below.sum()),
            "crossings": int(cross.sum()),
            "crossing_rate_%": 100 * float(cross.sum()) / max(int(below.sum()), 1),
        }
        for c in CEILINGS
        for below, cross in [M.crossing_labels(all_decision_rows, c)]
    ]
)
base_rates


In [ ]:
# Locate the first iteration whose sampled peak crosses each ceiling.
# For simulated ceilings, the trace records the phase of that iteration's peak;
# for the configured 500 MiB ceiling, it is the exact failing sample phase.
first_ceiling_breaks = pl.concat(
    [
        (
            iterations.filter(
                (pl.col("iteration_start_allocated") < ceiling)
                & (pl.col("iteration_peak_allocated") >= ceiling)
            )
            .sort("term", "iter_index")
            .group_by("term", maintain_order=True)
            .first()
            .select(
                pl.lit(ceiling / 2**20).alias("ceiling_mib"),
                "term",
                "term_size",
                pl.col("iter_index").alias("first_break_iteration"),
                pl.col("iteration_peak_phase").str.replace_all("_", " ").alias("break_phase"),
                pl.col("iteration_peak_rule").fill_null("—").alias("rule"),
                (pl.col("iteration_peak_allocated") / 2**20).alias("peak_memory_mib"),
            )
        )
        for ceiling in CEILINGS
    ],
    how="vertical",
)

break_phase_summary = (
    first_ceiling_breaks.group_by("ceiling_mib", "break_phase")
    .agg(
        pl.len().alias("runs"),
        pl.col("first_break_iteration").min().alias("first_iteration"),
        pl.col("first_break_iteration").median().alias("median_iteration"),
        pl.col("first_break_iteration").max().alias("last_iteration"),
    )
    .sort("ceiling_mib", "first_iteration")
)
display(break_phase_summary)

MP.ceiling_break_phase_chart(first_ceiling_breaks)
